In [ ]:
!pip install -q "sentence-transformers>=3.0.0"
!pip install -q rank_bm25
!pip install -q pyvi
!pip install -q faiss-gpu
import sentence_transformers, sys
print("sentence-transformers", sentence_transformers.__version__)
print("DONE")

In [ ]:
import os
import json 
import zipfile 
import time 
import glob 
import re 
import gc
import torch 
import numpy as np 

WORK_URL = "/kaggle/working"

def find_input(filename, required = True): 
    hits = sorted(glob.glob(f"/kaggle/input/**/{filename}", recursive = True))
    if not hits: 
        if required:
            raise FileNotFoundError(f"Không tìm thấy {filename} trong /kaggle/input")
        return None
    return hits[0]

TRAIN_URL  = find_input("train.json")
WARMUP_URL = find_input("warmup_IR.json")
PUBLIC_URL = find_input("public-official.json")
DEV_URL    = find_input("dev665.json", required = False)
# dev665 = 665 cau co split="dev" luc mining -> reranker CHUA BAO GIO train len chung.
# La tap do noi bo: thieu file nay van chay binh thuong va van ra submission.
EVAL_SET = "both"
EVAL_SETS = []
if EVAL_SET in ("warmup", "both"):
    EVAL_SETS.append(("warmup", WARMUP_URL))
if EVAL_SET in ("dev", "dev665", "both"):
    if DEV_URL:
        EVAL_SETS.append(("dev665", DEV_URL))
    else:
        print("  (khong co dev665.json trong Input -> chi danh gia tren warmup.")
        print("   dev665 la tap do noi bo, KHONG can de tao submission.)")
if not EVAL_SETS:
    raise RuntimeError(f"EVAL_SET = {EVAL_SET!r} khong chon duoc tap nao")
print("EVAL_SETS:", ", ".join(n for n, _ in EVAL_SETS))

CORPUS_FILES_URL = glob.glob("/kaggle/input/**/context_*.json", recursive = True)
if not CORPUS_FILES_URL: 
    raise FileNotFoundError("Không tìm thấy context_*.json trong /kaggle/input")

print("TRAIN:", TRAIN_URL)
print("WARMUP:", WARMUP_URL)
print("PUBLIC:", PUBLIC_URL)
print("CORPUS:", os.path.dirname(CORPUS_FILES_URL[0]), "|", len(CORPUS_FILES_URL), "files")

EMB_MODEL    = "AITeamVN/Vietnamese_Embedding_v2"
_ft = sorted(glob.glob("/kaggle/input/**/reranker_ft", recursive = True))
if not _ft:
    _ft = sorted({os.path.dirname(p) for p in
                  glob.glob("/kaggle/input/**/model.safetensors", recursive = True)})
if len(_ft) > 1:
    raise RuntimeError("Input dang co NHIEU HON MOT model: " + " | ".join(_ft)
                       + "  -> Go bot, chi giu DUNG MOT ban reranker (pool_n10).")
RERANK_MODEL = _ft[0] if _ft else "AITeamVN/Vietnamese_Reranker"
print("RERANK_MODEL:", RERANK_MODEL)
if not _ft:
    print("  !! CANH BAO: dang dung reranker GOC, chua fine-tune.")

# Hypermeter 
CHUNK_WORDS    = 220 
CHUNK_OVERLAP  = 40 
DENSE_TOPK     = 100
BM25_TOPK      = 100
RRF_K          = 60
DOC_CANDIDATES = 30    
FINAL_K        = 5 
CHUNKS_PER_DOC = 7
#   "max"  : diem doc = chunk RRF cao nhat 
#   "sum3" : tong RRF cua 3 chunk cao nhat -> doc co NHIEU doan khop duoc uu tien
DOC_RANK_MODE  = "max"
EMB_BATCH      = 128   # ha tu 256: o 512 token + fp16, batch 256 de OOM tren T4 16GB
RERANK_BATCH   = 128
MAX_SEQ_LEN    = 512

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| GPUs", torch.cuda.device_count())

In [ ]:
def load_corpus(files): 
    docs = {}
    for fp in files: 
        with open(fp, encoding = "utf-8") as f: 
            o = json.load(f)
        docs[str(o["id"])] = {"name": o.get("name", ""),"passage": o.get("passage", "")} 
    return docs 
DOCS = load_corpus(CORPUS_FILES_URL)
print(f"{len(DOCS)} văn bản")

8532 văn bản


In [ ]:
DIEU_RE = re.compile(r"(?=Điều\s+\d+[\.\:]?)")

def sliding(words, size, overlap): 
    step = max(1,size - overlap) 
    for i in range(0, len(words), step): 
        seg = words[i:i+size]
        if seg: 
            yield " ".join(seg)
        if i + size >= len(words): 
            break 

def chunk_doc(name, passage, size = CHUNK_WORDS, overlap = CHUNK_OVERLAP): 
    text = re.sub(r"\s+", " ", passage).strip()
    parts = [p.strip() for p in DIEU_RE.split(text) if p.strip()]
    if len(parts) <= 1: 
        parts = list(sliding(text.split(), size, overlap))
    chunks = []
    for p in parts: 
        w = p.split()
        if len(w) <= size*1.5: 
            chunks.append(p)
        else: 
            chunks.extend(sliding(w, size, overlap))
    prefix = (name.replace("-", " ") + " . ") if name else ""
    return [prefix + c for c in chunks] if chunks else [prefix]

chunk_texts, chunk_doc_ids = [], []
for did, d in DOCS.items(): 
    for c in chunk_doc(d["name"], d["passage"]): 
        chunk_texts.append(c) 
        chunk_doc_ids.append(did)
chunk_doc_ids = np.array(chunk_doc_ids)
print(f"{len(chunk_texts)} chunk từ {len(DOCS)} docs")

In [ ]:
from array import array

USE_PYVI = False 
_tok_re = re.compile(r"[0-9a-zà-ỹ/\-]+", re.IGNORECASE | re.UNICODE)
if USE_PYVI: 
    from pyvi import ViTokenizer
    def vi_tokenizer(text): 
        return _tok_re.findall(ViTokenizer.tokenizer(text.lower()))
else: 
    def vi_tokenizer(text): 
        return _tok_re.findall(text.lower())


class SparseBM25: 
    """BM25 Okapi trên inverted index numpy — thay cho rank_bm25.BM25Okapi."""

    def __init__(self, tokenized_corpus, k1 = 1.5, b = 0.75, verbose = False): 
        vocab = {}
        rows, cols, tfs, dl = array("i"), array("i"), array("f"), array("f")
        for i, toks in enumerate(tokenized_corpus): 
            dl.append(len(toks))
            if toks: 
                cnt = {}
                for t in toks: 
                    ti = vocab.get(t)
                    if ti is None: 
                        ti = len(vocab); vocab[t] = ti
                    cnt[ti] = cnt.get(ti, 0) + 1
                for ti, c in cnt.items(): 
                    rows.append(i); cols.append(ti); tfs.append(c)
            if verbose and (i + 1) % 100000 == 0: 
                print(f"    index {i+1:,} chunk, vocab {len(vocab):,}", flush = True)

        self.vocab = vocab
        self.n_docs = len(dl)
        dl = np.frombuffer(dl, dtype = np.float32)
        avgdl = float(dl.mean()) if self.n_docs else 1.0

        rows = np.frombuffer(rows, dtype = np.int32)
        cols = np.frombuffer(cols, dtype = np.int32)
        tfs  = np.frombuffer(tfs,  dtype = np.float32)

        order      = np.argsort(cols, kind = "stable")   # gom postings theo term
        self.p_doc = rows[order]
        tf_sorted  = tfs[order]
        col_sorted = cols[order]

        self.term_ptr = np.searchsorted(col_sorted, np.arange(len(vocab) + 1)).astype(np.int64)
        df = np.diff(self.term_ptr).astype(np.float64)
        idf = np.log(1.0 + (self.n_docs - df + 0.5) / (df + 0.5)).astype(np.float32)

        # trọng số BM25 không phụ thuộc query -> tính sẵn một lần
        norm = (1 - b + b * dl[self.p_doc] / avgdl).astype(np.float32)
        self.p_w = (tf_sorted * (k1 + 1) / (tf_sorted + k1 * norm)).astype(np.float32)
        self.p_w *= np.repeat(idf, np.diff(self.term_ptr))
        del rows, cols, tfs, order, tf_sorted, col_sorted, norm

    def get_scores(self, query_tokens): 
        score = np.zeros(self.n_docs, dtype = np.float32)
        for t in query_tokens: 
            ti = self.vocab.get(t)
            if ti is None: 
                continue
            s, e = self.term_ptr[ti], self.term_ptr[ti + 1]
            if s < e: 
                score[self.p_doc[s:e]] += self.p_w[s:e]
        return score

    @property
    def nnz(self): 
        return len(self.p_doc)


t0 = time.time()
# Truyen GENERATOR chu khong phai list: 133 trieu token khong bao gio nam cung luc trong RAM.
bm25 = SparseBM25((vi_tokenizer(t) for t in chunk_texts), verbose = True)
gc.collect()
print(f"BM25 index xong: {bm25.n_docs:,} chunk | vocab {len(bm25.vocab):,} | "
      f"postings {bm25.nnz:,} | {time.time()-t0:.0f}s (USE_PYVI = {USE_PYVI})")
try: 
    import subprocess
    print(subprocess.run(["free","-g"], capture_output=True, text=True).stdout)
except Exception: 
    pass

In [ ]:
import faiss 
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer(EMB_MODEL, device = DEVICE,
                                model_kwargs = {"torch_dtype": torch.float16})
emb_model.max_seq_length = MAX_SEQ_LEN
print("dtype:", next(emb_model.parameters()).dtype, "| max_seq_length:", emb_model.max_seq_length)

def encode_all(texts): 
    kw = dict(batch_size = EMB_BATCH, normalize_embeddings = True,
              convert_to_numpy = True, show_progress_bar = True)
    n_gpu = torch.cuda.device_count()
    if n_gpu > 1: 
        devs = [f"cuda:{i}" for i in range(n_gpu)]
        try: 
            out = emb_model.encode(texts, device = devs, **kw)
            print(f"  -> encode(device=list) trên {n_gpu} GPU")
            return out
        except Exception as e: 
            print("  [1] encode(device=list) không dùng được:", type(e).__name__, e)
        try: 
            pool = emb_model.start_multi_process_pool(target_devices = devs)
            try: 
                out = emb_model.encode_multi_process(
                    texts, pool, batch_size = EMB_BATCH, normalize_embeddings = True)
            finally: 
                emb_model.stop_multi_process_pool(pool)
            print(f"  -> encode_multi_process trên {n_gpu} GPU")
            return out
        except Exception as e: 
            print("  [2] start_multi_process_pool không dùng được:", type(e).__name__, e)
        print("  -> rơi về 1 GPU (vẫn fp16, chỉ chậm hơn ~2x)")
    return emb_model.encode(texts, **kw)

_cname = f"chunk_emb_L{MAX_SEQ_LEN}_w{CHUNK_WORDS}o{CHUNK_OVERLAP}.npy"
CACHE = os.path.join(WORK_URL, _cname)
if not os.path.exists(CACHE):
    _hit = sorted(glob.glob(f"/kaggle/input/**/{_cname}", recursive = True))
    if _hit:
        CACHE = _hit[0]
        print("Dung embedding san co tu Input:", CACHE)
if os.path.exists(CACHE): 
    chunk_emb = np.load(CACHE)
    print("Nạp embedding từ cache", os.path.basename(CACHE), chunk_emb.shape)
    if chunk_emb.shape[0] != len(chunk_texts): 
        raise RuntimeError(
            f"Cache {chunk_emb.shape[0]} vector nhưng đang có {len(chunk_texts)} chunk. "
            "Chunker đã đổi -> xoá file cache này rồi chạy lại.")
else: 
    t0 = time.time()
    chunk_emb = np.asarray(encode_all(chunk_texts)).astype("float32")  # FAISS cần float32
    np.save(CACHE, chunk_emb)
    print(f"Embed {len(chunk_texts)} chunk {time.time() -t0:.1f}s, shape = {chunk_emb.shape}")

index = faiss.IndexFlatIP(chunk_emb.shape[1])
index.add(chunk_emb)

# IndexFlatIP mac dinh o CPU: moi query quet tuan tu 2,2 GB vector (~0,5s/cau). Len GPU con vai ms.
try: 
    _res = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(_res, 0, index)
    print("FAISS: da chuyen len GPU")
except Exception as e: 
    print("FAISS van o CPU:", type(e).__name__, e)

del chunk_emb   # 2,2 GB, index da giu ban sao roi; can cache thi doc lai tu .npy
gc.collect()
print("FAISS ntotal:", index.ntotal)

In [ ]:
def rrf_fuse(dense_ids, bm25_ids, k= RRF_K): 
    score = {}
    for rank, cid in enumerate(dense_ids): 
        score[cid] = score.get(cid, 0) + 1/(k+rank)
    for rank, cid in enumerate(bm25_ids): 
        score[cid] = score.get(cid, 0) + 1/(k+rank)
    return score 

def retrieve_doc_candidate(query, n_docs = None, n_chunks = None): 
    n_docs   = DOC_CANDIDATES  if n_docs   is None else n_docs
    n_chunks = CHUNKS_PER_DOC  if n_chunks is None else n_chunks
    qv = emb_model.encode([query], normalize_embeddings = True, 
                         convert_to_numpy = True).astype("float32")
    _, dense_idx = index.search(qv, DENSE_TOPK)
    dense_idx = dense_idx[0]
    bm25_scores = bm25.get_scores(vi_tokenizer(query))
    bm25_idx = np.argpartition(bm25_scores, -BM25_TOPK)[-BM25_TOPK:]
    bm25_idx = bm25_idx[np.argsort(bm25_scores[bm25_idx])[::-1]]
    fused = rrf_fuse(list(dense_idx), list(bm25_idx))

    doc_chunks = {}
    for cid, s in fused.items(): 
        doc_chunks.setdefault(chunk_doc_ids[cid], []).append((s, cid))
    # Thu hang doc = chunk tot nhat cua no -> KHONG phu thuoc n_chunks (cell 9 dua vao dieu nay).
    if DOC_RANK_MODE == "sum3":
        key = lambda kv: sum(s for s, _ in sorted(kv[1], reverse = True)[:3])
    else:
        key = lambda kv: max(kv[1])[0]
    ranked = sorted(doc_chunks.items(), key = key, reverse = True)[:n_docs]
    return [(did, [cid for _, cid in sorted(lst, reverse = True)[:n_chunks]])
            for did, lst in ranked]   # (doc_id, [chunk_id, ...])


In [ ]:
from sentence_transformers import CrossEncoder 
reranker = CrossEncoder(RERANK_MODEL, device = DEVICE, max_length = 512 )

# fp32 tren T4 mat ~2,5s/cau, fp16 con ~0,4s/cau. Dung .model.half() thay vi tham so dtype
# vi ten tham so cua CrossEncoder doi giua cac ban sentence-transformers.
if DEVICE == "cuda": 
    reranker.model.half()
print("reranker dtype:", next(reranker.model.parameters()).dtype)

def rerank_top5 (query, candidates): 
    if not candidates: 
        return []
    pairs, owner = [], []
    for di, (did, cids) in enumerate(candidates): 
        for cid in cids: 
            pairs.append([query, chunk_texts[cid]])
            owner.append(di)
    # show_progress_bar=False: bat len thi moi cau in mot thanh tien trinh, ngap output.
    scores = np.asarray(reranker.predict(pairs, batch_size = RERANK_BATCH,
                                         show_progress_bar = False), dtype = np.float32)
    # Diem doc = max tren cac chunk cua no.
    best = np.full(len(candidates), -np.inf, dtype = np.float32)
    np.maximum.at(best, np.asarray(owner), scores)
    order = np.argsort(best)[::-1][:FINAL_K]
    return [candidates[i][0] for i in order]

def predict(query): 
    return rerank_top5(query, retrieve_doc_candidate(query))


In [ ]:
# eval_official: sao y data/Scoring-Program-Task-LegalIR/scoring.py cua BTC.
# Giu du 2 bay: len(pred)>5 hoac pred rong -> 0 diem; lech so cau -> raise.
def eval_official(y_pred, y_true, max_pred = FINAL_K):
    if len(y_pred) != len(y_true):
        raise Exception(f"Samples in predict not match with reference: {len(y_pred)} vs {len(y_true)}")
    rec, pre = [], []
    for qid, v in y_true.items():
        gold = {str(x) for x in v["answer"]}
        p = [str(x) for x in y_pred.get(str(qid), {}).get("answer", [])]
        if len(p) == 0 or len(p) > max_pred:
            rec.append(0.0); pre.append(0.0); continue
        hit = len(gold & set(p))
        rec.append(hit/len(gold) if gold else 0.0)
        pre.append(hit/len(p))
    return {"recall": sum(rec)/len(rec), "precision": sum(pre)/len(pre), "n": len(rec)}


RUN_TAG = f"L{MAX_SEQ_LEN}_c{BM25_TOPK}_d{DOC_CANDIDATES}_k{CHUNKS_PER_DOC}_{DOC_RANK_MODE}"
EVAL_RESULTS = {}

for SET_NAME, SET_URL in EVAL_SETS:
    data = json.load(open(SET_URL, encoding = "utf-8"))
    t0 = time.time()
    preds, pool_hit = {}, []
    for i, (qid, it) in enumerate(data.items()):
        # Tach 2 buoc de do luon TRAN RO: gold co nam trong DOC_CANDIDATES ung vien khong.
        cands = retrieve_doc_candidate(it["question"])
        gold  = {str(x) for x in it["answer"]}
        pool  = {str(d) for d, _ in cands}
        pool_hit.append(len(gold & pool)/len(gold) if gold else 0.0)
        preds[str(qid)] = {"answer": [str(d) for d in rerank_top5(it["question"], cands)]}
        if (i+1) % 100 == 0:
            el = time.time() - t0
            print(f"  [{SET_NAME}] {i+1}/{len(data)}  {el/60:.1f}p"
                  f"  (con ~{el/(i+1)*(len(data)-i-1)/60:.1f}p)", flush = True)
    ceiling = sum(pool_hit)/len(pool_hit)

    res = eval_official(preds, data)
    n_bad = sum(1 for v in preds.values() if not (1 <= len(v["answer"]) <= FINAL_K))
    # tran ro thap -> loi o RETRIEVAL; ti le nhat thap -> loi o RERANKER.
    print(f"\n=== {SET_NAME.upper()}  |  {res['n']} cau  |  L{MAX_SEQ_LEN} ===")
    print(f"  Recall@5 {res['recall']:.4f} | Precision {res['precision']:.4f}"
          f" | tra sai so luong: {n_bad}")
    print(f"  tran ro {ceiling:.4f} | nhat duoc {res['recall']/ceiling*100:.1f}%"
          f" | {(time.time()-t0)/60:.1f} phut")

    EVAL_RESULTS[SET_NAME] = {"ceiling": ceiling, **res}
    ev = os.path.join(WORK_URL, f"{SET_NAME}_eval_{RUN_TAG}.json")
    json.dump({"eval_set": SET_NAME, "max_seq_len": MAX_SEQ_LEN, "chunk_words": CHUNK_WORDS,
               "bm25_topk": BM25_TOPK, "doc_candidates": DOC_CANDIDATES,
               "chunks_per_doc": CHUNKS_PER_DOC, "doc_rank_mode": DOC_RANK_MODE,
               "ceiling": ceiling, **res},
              open(ev, "w", encoding = "utf-8"), ensure_ascii = False, indent = 2)
    # Du doan TUNG CAU -> so sanh theo cap giua cac lan chay, nhay hon han so trung binh.
    pj = os.path.join(WORK_URL, f"{SET_NAME}_pred_{RUN_TAG}.json")
    json.dump(preds, open(pj, "w", encoding = "utf-8"), ensure_ascii = False)
    print(f"  -> {os.path.basename(ev)}  {os.path.basename(pj)}")

# warmup va dev665 la hai tap DOC LAP: mot cai tien chi len o mot tap thi chua ket luan duoc.


In [ ]:
# ================== CELL CHAN DOAN ======================
import gzip as _gz

DIAG_KMAX  = 12
DIAG_TOPKS = [100]           # 100 = cau hinh dang dung. 300 da do, khong an diem.


def retrieve_diag(query, topk, n_docs, n_chunks):
    """Ban sao retrieve_doc_candidate(), them topk tu do va so chunk that co moi doc.

    Xep doc bang max(kv[1]) = so ca tuple (diem, cid), con ban that dung max(kv[1])[0]
    = chi so diem. Hoa diem RRF hay xay ra nen hai cach pha hoa khac nhau -> lech ~1 cau
    tren 500. So sanh BEN TRONG cung mot dump thi khong bi anh huong.
    """
    qv = emb_model.encode([query], normalize_embeddings = True,
                          convert_to_numpy = True).astype("float32")
    _, di = index.search(qv, topk)
    bs = bm25.get_scores(vi_tokenizer(query))
    bi = np.argpartition(bs, -topk)[-topk:]
    bi = bi[np.argsort(bs[bi])[::-1]]
    fused = rrf_fuse(list(di[0]), list(bi))

    doc_chunks = {}
    for cid, s in fused.items():
        doc_chunks.setdefault(chunk_doc_ids[cid], []).append((s, cid))
    ranked = sorted(doc_chunks.items(), key = lambda kv: max(kv[1]), reverse = True)[:n_docs]
    # tra ve (doc_id, chunk da cat theo n_chunks, TONG so chunk doc do co trong ro)
    return [(did, [c for _, c in sorted(l, reverse = True)[:n_chunks]], len(l))
            for did, l in ranked]


def eval_at_k(diag, k, final_k = FINAL_K):
    rec, pre = [], []
    for d in diag:
        best  = [max(v[:k]) if v else -1e9 for v in d["scores"]]
        order = np.argsort(best)[::-1][:final_k]
        pred  = {d["pool"][j] for j in order}
        g     = set(d["gold"])
        rec.append(len(g & pred) / len(g) if g else 0.0)
        pre.append(len(g & pred) / final_k)
    return float(np.mean(rec)), float(np.mean(pre))


for SET_NAME, SET_URL in EVAL_SETS:
    data = json.load(open(SET_URL, encoding = "utf-8"))
    for TOPK in DIAG_TOPKS:
        TAG = f"{SET_NAME}_L{MAX_SEQ_LEN}_c{TOPK}_d{DOC_CANDIDATES}_kmax{DIAG_KMAX}_{DOC_RANK_MODE}"
        print(f"\n### {SET_NAME}  |  chunk-topk {TOPK}  |  doc {DOC_CANDIDATES}"
              f"  |  kmax {DIAG_KMAX}  |  {len(data)} cau", flush = True)

        diag, n_pairs = [], 0
        t0 = time.time()
        for i, (qid, it) in enumerate(data.items()):
            q     = it["question"]
            cands = retrieve_diag(q, TOPK, DOC_CANDIDATES, DIAG_KMAX)

            pairs, owner = [], []
            for di_, (did, cids, _) in enumerate(cands):
                for cid in cids:
                    pairs.append([q, chunk_texts[cid]]); owner.append(di_)
            n_pairs += len(pairs)
            sc = np.asarray(reranker.predict(pairs, batch_size = RERANK_BATCH,
                                             show_progress_bar = False), dtype = np.float32)
            per_doc = [[] for _ in cands]
            for o, s in zip(owner, sc):
                per_doc[o].append(round(float(s), 4))   # giu nguyen thu tu RRF giam dan

            diag.append({"qid": str(qid), "question": q,
                         "gold":   sorted({str(x) for x in it["answer"]}),
                         "pool":   [str(d) for d, _, _ in cands],
                         "cids":   [[int(c) for c in cs] for _, cs, _ in cands],
                         "supply": [int(n) for _, _, n in cands],
                         "scores": per_doc})
            if (i + 1) % 100 == 0:
                el = time.time() - t0
                print(f"  {i+1}/{len(data)}  {el/60:.1f}p"
                      f"  (con ~{el/(i+1)*(len(data)-i-1)/60:.1f}p)", flush = True)

        # --- nguon cung chunk: k co bi CHAN TREN khong ---------------------------
        sup = np.array([n for d in diag for n in d["supply"]])
        print(f"\n  da cham {n_pairs:,} cap = {n_pairs/len(data):.0f} cap/cau"
              f"  ({(time.time()-t0)/60:.0f} phut)")
        print(f"  chunk moi doc trong ro: median {np.median(sup):.0f}  mean {sup.mean():.2f}"
              f"  p90 {np.percentile(sup,90):.0f}  max {sup.max()}")
        print("  ti le doc co DU k chunk: " +
              "  ".join(f"k={k}:{(sup>=k).mean()*100:.0f}%" for k in (1, 3, 5, 7, 10, 12)))

        # --- quet k tu chinh du lieu vua cham ------------------------------------
        ceiling = float(np.mean([len(set(d["gold"]) & set(d["pool"])) / len(d["gold"])
                                 for d in diag]))
        print(f"\n  tran ro {ceiling:.4f}   |   doc unique trung binh "
              f"{np.mean([len(d['pool']) for d in diag]):.1f}")
        print("   k    Recall@5   Precision    Delta    % nhat tu ro")
        sweep, prev = {}, None
        for k in range(1, DIAG_KMAX + 1):
            r, p = eval_at_k(diag, k); sweep[k] = {"recall": r, "precision": p}
            print(f"  {k:>2}    {r:.4f}     {p:.4f}   "
                  f"{'   --  ' if prev is None else f'{r-prev:+.4f}'}   {r/ceiling*100:5.1f}%")
            prev = r

        json.dump({"tag": TAG, "eval_set": SET_NAME, "chunk_topk": TOPK,
                   "doc_candidates": DOC_CANDIDATES, "ceiling": ceiling,
                   "n": len(diag), "n_pairs": n_pairs,
                   "supply": {"median": float(np.median(sup)), "mean": float(sup.mean()),
                              "p90": float(np.percentile(sup, 90)), "max": int(sup.max()),
                              "enough_k": {k: float((sup >= k).mean())
                                           for k in range(1, DIAG_KMAX + 1)}},
                   "sweep": sweep},
                  open(os.path.join(WORK_URL, f"ksweep_{TAG}.json"), "w", encoding = "utf-8"),
                  ensure_ascii = False, indent = 2)

        # --- dump chan doan: kem VAN BAN chunk chi cho nhung cau CON SAI ---------
        K_NOW = CHUNKS_PER_DOC
        need = set()
        for d in diag:
            best  = [max(v[:K_NOW]) if v else -1e9 for v in d["scores"]]
            order = [int(j) for j in np.argsort(best)[::-1]]
            d["order"] = order
            d["pred"]  = [d["pool"][j] for j in order[:FINAL_K]]
            d["recall_now"] = len(set(d["gold"]) & set(d["pred"])) / len(d["gold"])
            if d["recall_now"] < 1.0:
                need.update((d["qid"], j) for j in order[:FINAL_K])
                need.update((d["qid"], j) for j, did in enumerate(d["pool"]) if did in set(d["gold"]))
        for d in diag:
            d["texts"] = {str(j): [chunk_texts[c][:600] for c in d["cids"][j]]
                          for j in range(len(d["pool"])) if (d["qid"], j) in need}

        dj = os.path.join(WORK_URL, f"diag_{TAG}.json.gz")
        with _gz.open(dj, "wt", encoding = "utf-8") as f:
            json.dump(diag, f, ensure_ascii = False)
        print(f"\n  {sum(1 for d in diag if d['recall_now'] < 1.0)} cau con sai o k={K_NOW}"
              f"  ->  {dj}  ({os.path.getsize(dj)/1e6:.1f} MB)")

print("\nXONG. Quick Save, tai ve may: ksweep_*.json + diag_*.json.gz + *_pred_*.json")


In [ ]:
# --- Submission --- 
public = json.load(open(PUBLIC_URL, encoding = "utf-8"))
sub = {}
t0 = time.time()
for i, (qid, it) in enumerate(public.items()): 
    sub[qid] = {"answer": predict(it["question"])}
    if (i+1)%100 ==0: 
        print(f"{i+1}/{len(public)}")
sj = os.path.join(WORK_URL, "submission.json")
json.dump(sub, open(sj, "w", encoding = "utf-8"), ensure_ascii = False)
with zipfile.ZipFile(os.path.join(WORK_URL, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z: 
    z.write(sj, "submission.json")
print("DONE")